# Previsão diária de acidentes com ARIMAX

Este notebook agrega `Acidentes.csv` por dia e usa o volume diário de tráfego como variável externa de um modelo ARIMAX. No histórico, o modelo usa o volume observado. Para prever acidentes futuros, ele consome `previsao_volume_trafego.csv`, produzido pelo notebook `volume_trafego.ipynb`.

O alvo é transformado com `log1p` para manter as previsões não negativas e reduzir o efeito de dias extremos. O modelo também recebe componentes de calendário semanais e anuais. O intervalo de 95% dos acidentes é condicional à previsão central de tráfego e não incorpora toda a incerteza do SARIMAX.

In [ ]:
from pathlib import Path

_requirements = next(
    (path for path in [Path("../requirements.txt"), Path("RoadFixerAPI/requirements.txt")] if path.exists()),
    None,
)
if _requirements is None:
    raise FileNotFoundError("requirements.txt não encontrado.")
_requirements = str(_requirements)
%pip install -r $_requirements

In [ ]:
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

HORIZONTE_VALIDACAO = 30
JANELA_TREINO_DIAS = 730
ORDEM_ARIMAX = (1, 0, 1)
LIMIAR_DIA_INCOMPLETO = 0.60
COLUNAS_VOLUME = ["QTD_MOTO", "QTD_PASSEIO", "QTD_COMERCIAL"]

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Localização e validação dos arquivos

In [ ]:
def localizar_arquivo(caminhos_relativos, descricao):
    cwd = Path.cwd().resolve()
    bases = [cwd, cwd.parent]
    candidatos = []
    for base in bases:
        for relativo in caminhos_relativos:
            candidato = (base / relativo).resolve()
            if candidato not in candidatos:
                candidatos.append(candidato)

    for candidato in candidatos:
        if candidato.is_file():
            return candidato

    verificados = "\n".join(f"- {path}" for path in candidatos)
    raise FileNotFoundError(f"{descricao} não encontrado. Caminhos verificados:\n{verificados}")


ARQUIVO_ACIDENTES = localizar_arquivo(
    [
        Path("Database/In/acidentes.csv"),
        Path("Database/In/Acidentes.csv"),
        Path("Database/Processed/acidentes.csv"),
        Path("Database/Processed/Acidentes.csv"),
        Path("RoadFixerAPI/Database/In/acidentes.csv"),
        Path("RoadFixerAPI/Database/In/Acidentes.csv"),
        Path("RoadFixerAPI/Database/Processed/acidentes.csv"),
        Path("RoadFixerAPI/Database/Processed/Acidentes.csv"),
    ],
    "Acidentes.csv",
)
ARQUIVO_VOLUME_HISTORICO = localizar_arquivo(
    [
        Path("Database/In/volumediario.csv"),
        Path("Database/In/VolumeDiario.csv"),
        Path("Database/Processed/volumediario.csv"),
        Path("Database/Processed/VolumeDiario.csv"),
        Path("RoadFixerAPI/Database/In/volumediario.csv"),
        Path("RoadFixerAPI/Database/In/VolumeDiario.csv"),
        Path("RoadFixerAPI/Database/Processed/volumediario.csv"),
        Path("RoadFixerAPI/Database/Processed/VolumeDiario.csv"),
    ],
    "VolumeDiario.csv",
)
ARQUIVO_VOLUME_PREVISTO = localizar_arquivo(
    [
        Path("artifacts/previsao_volume_trafego.csv"),
        Path("ModeloEstatistico/artifacts/previsao_volume_trafego.csv"),
        Path("RoadFixerAPI/ModeloEstatistico/artifacts/previsao_volume_trafego.csv"),
    ],
    "previsao_volume_trafego.csv; execute primeiro o notebook volume_trafego.ipynb",
)

print(f"Acidentes: {ARQUIVO_ACIDENTES}")
print(f"Volume histórico: {ARQUIVO_VOLUME_HISTORICO}")
print(f"Volume previsto pelo SARIMAX: {ARQUIVO_VOLUME_PREVISTO}")

In [ ]:
acidentes_brutos = pd.read_csv(ARQUIVO_ACIDENTES, encoding="utf-8-sig", low_memory=False)
volume_bruto = pd.read_csv(ARQUIVO_VOLUME_HISTORICO, encoding="utf-8-sig")
volume_previsto_bruto = pd.read_csv(ARQUIVO_VOLUME_PREVISTO, encoding="utf-8-sig")

for tabela in [acidentes_brutos, volume_bruto, volume_previsto_bruto]:
    tabela.columns = tabela.columns.str.strip().str.upper()

if "DATA" not in acidentes_brutos.columns:
    raise ValueError("Acidentes.csv precisa conter a coluna DATA.")

colunas_volume_ausentes = sorted({"DATA", *COLUNAS_VOLUME} - set(volume_bruto.columns))
if colunas_volume_ausentes:
    raise ValueError(f"Colunas ausentes em VolumeDiario.csv: {colunas_volume_ausentes}")

colunas_previsao_ausentes = sorted(
    {"DATA", "PREVISAO_VOLUME"} - set(volume_previsto_bruto.columns)
)
if colunas_previsao_ausentes:
    raise ValueError(
        f"Colunas ausentes em previsao_volume_trafego.csv: {colunas_previsao_ausentes}"
    )

print(f"Ocorrências carregadas: {len(acidentes_brutos):,}")
print(f"Registros de volume carregados: {len(volume_bruto):,}")
print(f"Dias de volume previsto: {len(volume_previsto_bruto):,}")

## 2. Agregação diária e alinhamento das séries

In [ ]:
acidentes = acidentes_brutos.copy()
acidentes["DATA"] = pd.to_datetime(
    acidentes["DATA"], format="mixed", errors="coerce"
)
datas_invalidas = int(acidentes["DATA"].isna().sum())
if datas_invalidas:
    warnings.warn(f"{datas_invalidas} ocorrências com DATA inválida foram removidas.")
acidentes = acidentes.dropna(subset=["DATA"])

acidentes_diarios = acidentes.groupby("DATA").size().rename("ACIDENTES")
indice_acidentes = pd.date_range(
    acidentes_diarios.index.min(), acidentes_diarios.index.max(), freq="D"
)
acidentes_diarios = acidentes_diarios.reindex(indice_acidentes, fill_value=0).asfreq("D")
acidentes_diarios.index.name = "DATA"

print(
    f"Acidentes disponíveis de {acidentes_diarios.index.min().date()} "
    f"a {acidentes_diarios.index.max().date()}"
)
print(f"Média diária: {acidentes_diarios.mean():.2f} acidentes")
acidentes_diarios.tail()

In [ ]:
volume = volume_bruto.copy()
volume["DATA"] = pd.to_datetime(volume["DATA"], format="mixed", errors="coerce")
for coluna in COLUNAS_VOLUME:
    volume[coluna] = pd.to_numeric(volume[coluna], errors="coerce")

volume_invalido = volume[["DATA", *COLUNAS_VOLUME]].isna().any(axis=1)
volume_negativo = (volume[COLUNAS_VOLUME] < 0).any(axis=1)
quantidade_volume_invalido = int((volume_invalido | volume_negativo).sum())
if quantidade_volume_invalido:
    warnings.warn(f"{quantidade_volume_invalido} registros de volume inválidos foram removidos.")
volume = volume.loc[~(volume_invalido | volume_negativo)].copy()
volume["VOLUME_TOTAL"] = volume[COLUNAS_VOLUME].sum(axis=1)
volume_diario = volume.groupby("DATA")["VOLUME_TOTAL"].sum().sort_index()

ultima_data_volume_descartada = None
if len(volume_diario) >= 29:
    referencia_recente = volume_diario.iloc[-29:-1].median()
    proporcao_ultimo_dia = volume_diario.iloc[-1] / referencia_recente
    if proporcao_ultimo_dia < LIMIAR_DIA_INCOMPLETO:
        ultima_data_volume_descartada = volume_diario.index[-1]
        warnings.warn(
            f"O último dia de tráfego ({ultima_data_volume_descartada.date()}) tem apenas "
            f"{proporcao_ultimo_dia:.1%} da mediana recente e foi tratado como incompleto."
        )
        volume_diario = volume_diario.iloc[:-1]

indice_volume = pd.date_range(volume_diario.index.min(), volume_diario.index.max(), freq="D")
volume_diario = volume_diario.reindex(indice_volume)
dias_volume_ausentes = int(volume_diario.isna().sum())
if dias_volume_ausentes:
    warnings.warn(f"{dias_volume_ausentes} dias de volume serão interpolados.")
    volume_diario = volume_diario.interpolate(limit_direction="both")
volume_diario = volume_diario.rename("VOLUME_TOTAL").asfreq("D")
volume_diario.index.name = "DATA"

volume_previsto = volume_previsto_bruto[["DATA", "PREVISAO_VOLUME"]].copy()
volume_previsto["DATA"] = pd.to_datetime(
    volume_previsto["DATA"], format="mixed", errors="coerce"
)
volume_previsto["PREVISAO_VOLUME"] = pd.to_numeric(
    volume_previsto["PREVISAO_VOLUME"], errors="coerce"
)
volume_previsto = (
    volume_previsto.dropna()
    .loc[lambda df: df["PREVISAO_VOLUME"] > 0]
    .drop_duplicates(subset="DATA", keep="last")
    .sort_values("DATA")
    .set_index("DATA")
)

if volume_previsto.empty:
    raise ValueError("A previsão SARIMAX não contém volumes válidos.")
if volume_previsto.index.min() != volume_diario.index.max() + pd.Timedelta(days=1):
    raise ValueError(
        "A previsão SARIMAX precisa começar no dia seguinte ao último volume histórico válido. "
        f"Histórico termina em {volume_diario.index.max().date()} e a previsão começa em "
        f"{volume_previsto.index.min().date()}."
    )
indice_previsto_esperado = pd.date_range(
    volume_previsto.index.min(), volume_previsto.index.max(), freq="D"
)
if not volume_previsto.index.equals(indice_previsto_esperado):
    raise ValueError("A previsão SARIMAX contém lacunas entre as datas.")

historico = pd.concat([acidentes_diarios, volume_diario], axis=1, join="inner").dropna()
historico = historico.asfreq("D")
if historico.isna().any().any():
    raise ValueError("Há lacunas após alinhar acidentes e volume histórico.")
if len(historico) < JANELA_TREINO_DIAS + HORIZONTE_VALIDACAO:
    raise ValueError(
        f"São necessários ao menos {JANELA_TREINO_DIAS + HORIZONTE_VALIDACAO} dias alinhados, "
        f"mas foram encontrados {len(historico)}."
    )

print(
    f"Histórico alinhado: {historico.index.min().date()} a {historico.index.max().date()} "
    f"({len(historico):,} dias)"
)
print(
    f"Horizonte SARIMAX: {volume_previsto.index.min().date()} a "
    f"{volume_previsto.index.max().date()} ({len(volume_previsto)} dias)"
)
historico.tail()

In [ ]:
fig, eixos = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
eixos[0].plot(historico.index, historico["ACIDENTES"], color="#dc2626", linewidth=1)
eixos[0].set_title("Acidentes diários")
eixos[0].set_ylabel("Acidentes")
eixos[1].plot(historico.index, historico["VOLUME_TOTAL"], color="#2563eb", linewidth=1)
eixos[1].set_title("Volume diário de tráfego")
eixos[1].set_xlabel("Data")
eixos[1].set_ylabel("Veículos")
plt.tight_layout()
plt.show()

## 3. Funções do ARIMAX

In [ ]:
DATA_REFERENCIA = historico.index.min()


def calcular_escala_volume(volume_serie):
    volume_log = np.log1p(np.asarray(volume_serie, dtype=float))
    media = float(volume_log.mean())
    desvio = float(volume_log.std(ddof=1))
    if not np.isfinite(desvio) or desvio == 0:
        raise ValueError("O volume histórico não possui variação suficiente para o ARIMAX.")
    return media, desvio


def criar_variaveis_externas(indice, volumes, media_volume_log, desvio_volume_log):
    indice = pd.DatetimeIndex(indice)
    volumes = np.asarray(volumes, dtype=float)
    if len(indice) != len(volumes):
        raise ValueError("Datas e volumes precisam ter o mesmo tamanho.")
    if np.any(~np.isfinite(volumes)) or np.any(volumes <= 0):
        raise ValueError("Todos os volumes usados pelo ARIMAX devem ser positivos e finitos.")

    dia_semana = indice.dayofweek.to_numpy()
    dias = (indice - DATA_REFERENCIA).days.to_numpy()
    return pd.DataFrame(
        {
            "VOLUME_LOG_Z": (np.log1p(volumes) - media_volume_log) / desvio_volume_log,
            "SEMANA_SEN": np.sin(2 * np.pi * dia_semana / 7),
            "SEMANA_COS": np.cos(2 * np.pi * dia_semana / 7),
            "ANO_SEN": np.sin(2 * np.pi * dias / 365.25),
            "ANO_COS": np.cos(2 * np.pi * dias / 365.25),
        },
        index=indice,
    )


def ajustar_arimax(acidentes_serie, variaveis_externas):
    modelo = SARIMAX(
        np.log1p(acidentes_serie.astype(float)),
        exog=variaveis_externas,
        order=ORDEM_ARIMAX,
        seasonal_order=(0, 0, 0, 0),
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    return modelo.fit(disp=False, maxiter=200)


def voltar_escala_acidentes(valores):
    return np.clip(np.expm1(valores), a_min=0, a_max=None)

## 4. Validação temporal com volume observado

In [ ]:
dados_validacao = historico.tail(JANELA_TREINO_DIAS + HORIZONTE_VALIDACAO)
treino = dados_validacao.iloc[:-HORIZONTE_VALIDACAO]
teste = dados_validacao.iloc[-HORIZONTE_VALIDACAO:]
media_volume_validacao, desvio_volume_validacao = calcular_escala_volume(
    treino["VOLUME_TOTAL"]
)
exog_treino = criar_variaveis_externas(
    treino.index,
    treino["VOLUME_TOTAL"],
    media_volume_validacao,
    desvio_volume_validacao,
)
exog_teste = criar_variaveis_externas(
    teste.index,
    teste["VOLUME_TOTAL"],
    media_volume_validacao,
    desvio_volume_validacao,
)

modelo_validacao = ajustar_arimax(treino["ACIDENTES"], exog_treino)
resultado_validacao = modelo_validacao.get_forecast(
    steps=HORIZONTE_VALIDACAO, exog=exog_teste
)
previsao_validacao = pd.Series(
    voltar_escala_acidentes(resultado_validacao.predicted_mean.to_numpy()),
    index=teste.index,
    name="PREVISAO_ARIMAX",
)
observado = teste["ACIDENTES"]
baseline_semanal = dados_validacao["ACIDENTES"].shift(7).loc[teste.index]

mae = mean_absolute_error(observado, previsao_validacao)
rmse = np.sqrt(mean_squared_error(observado, previsao_validacao))
mape = np.abs((observado - previsao_validacao) / observado.clip(lower=1)).mean() * 100
mae_baseline = mean_absolute_error(observado, baseline_semanal)
metricas_validacao = pd.Series(
    {
        "MAE_ARIMAX": mae,
        "RMSE_ARIMAX": rmse,
        "MAPE_ARIMAX_PERCENTUAL": mape,
        "MAE_BASELINE_7_DIAS": mae_baseline,
        "GANHO_MAE_PERCENTUAL": (1 - mae / mae_baseline) * 100,
    }
)
display(metricas_validacao.to_frame("VALOR").round(2))

comparacao_validacao = pd.DataFrame(
    {
        "OBSERVADO": observado,
        "PREVISAO_ARIMAX": previsao_validacao,
        "BASELINE_7_DIAS": baseline_semanal,
    }
)
comparacao_validacao.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(comparacao_validacao.index, comparacao_validacao["OBSERVADO"], label="Observado", color="#111827", linewidth=2)
ax.plot(comparacao_validacao.index, comparacao_validacao["PREVISAO_ARIMAX"], label="Previsão ARIMAX", color="#dc2626", linewidth=2)
ax.plot(comparacao_validacao.index, comparacao_validacao["BASELINE_7_DIAS"], label="Baseline 7 dias", color="#9ca3af", linewidth=1)
ax.set_title("Validação temporal do ARIMAX")
ax.set_xlabel("Data")
ax.set_ylabel("Acidentes por dia")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Treino final e previsão com o volume do SARIMAX

In [ ]:
treino_final = historico.tail(JANELA_TREINO_DIAS)
media_volume_final, desvio_volume_final = calcular_escala_volume(
    treino_final["VOLUME_TOTAL"]
)
exog_final = criar_variaveis_externas(
    treino_final.index,
    treino_final["VOLUME_TOTAL"],
    media_volume_final,
    desvio_volume_final,
)
modelo_final = ajustar_arimax(treino_final["ACIDENTES"], exog_final)
coeficientes_modelo = pd.DataFrame(
    {"COEFICIENTE": modelo_final.params, "P_VALOR": modelo_final.pvalues}
)
display(coeficientes_modelo.round(4))

exog_futuro = criar_variaveis_externas(
    volume_previsto.index,
    volume_previsto["PREVISAO_VOLUME"],
    media_volume_final,
    desvio_volume_final,
)
resultado_futuro = modelo_final.get_forecast(
    steps=len(volume_previsto), exog=exog_futuro
)
previsao_acidentes_continua = voltar_escala_acidentes(
    resultado_futuro.predicted_mean.to_numpy()
)
intervalo_log = resultado_futuro.conf_int(alpha=0.05)
limite_inferior = voltar_escala_acidentes(intervalo_log.iloc[:, 0].to_numpy())
limite_superior = voltar_escala_acidentes(intervalo_log.iloc[:, 1].to_numpy())

previsao_acidentes = pd.DataFrame(
    {
        "PREVISAO_ACIDENTES": np.rint(previsao_acidentes_continua).astype(np.int64),
        "LIMITE_INFERIOR_95": np.floor(limite_inferior).astype(np.int64),
        "LIMITE_SUPERIOR_95": np.ceil(limite_superior).astype(np.int64),
        "VOLUME_PREVISTO_SARIMAX": volume_previsto["PREVISAO_VOLUME"].astype(np.int64),
    },
    index=volume_previsto.index,
)
previsao_acidentes.index.name = "DATA"
previsao_acidentes

## 6. Avaliação encadeada SARIMAX → ARIMAX, quando houver valores observados

In [ ]:
observado_futuro = acidentes_diarios.reindex(previsao_acidentes.index)
mascara_observada = observado_futuro.notna()
avaliacao_encadeada = previsao_acidentes.copy()
avaliacao_encadeada["ACIDENTES_OBSERVADOS"] = observado_futuro

metricas_encadeadas = None
if mascara_observada.any():
    y_real = observado_futuro.loc[mascara_observada]
    y_previsto = pd.Series(
        previsao_acidentes_continua, index=previsao_acidentes.index
    ).loc[mascara_observada]
    metricas_encadeadas = pd.Series(
        {
            "DIAS_AVALIADOS": int(mascara_observada.sum()),
            "MAE_ENCADEADO": mean_absolute_error(y_real, y_previsto),
            "RMSE_ENCADEADO": np.sqrt(mean_squared_error(y_real, y_previsto)),
            "MAPE_ENCADEADO_PERCENTUAL": (
                np.abs((y_real - y_previsto) / y_real.clip(lower=1)).mean() * 100
            ),
        }
    )
    display(metricas_encadeadas.to_frame("VALOR").round(2))
else:
    print("O horizonte previsto ainda não possui acidentes observados para avaliação.")

avaliacao_encadeada.tail()

In [ ]:
historico_grafico = historico["ACIDENTES"].tail(90)
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(historico_grafico.index, historico_grafico, label="Histórico usado no treino", color="#111827")
ax.plot(previsao_acidentes.index, previsao_acidentes_continua, label="Previsão ARIMAX", color="#dc2626", linewidth=2)
ax.fill_between(
    previsao_acidentes.index,
    limite_inferior,
    limite_superior,
    color="#fca5a5",
    alpha=0.35,
    label="Intervalo de 95%",
)
if mascara_observada.any():
    ax.plot(
        observado_futuro.index[mascara_observada],
        observado_futuro.loc[mascara_observada],
        label="Observado fora do treino",
        color="#2563eb",
        linewidth=1.5,
    )
ax.axvline(historico.index.max(), color="#6b7280", linestyle="--", linewidth=1)
ax.set_title("Previsão diária de acidentes usando o volume previsto pelo SARIMAX")
ax.set_xlabel("Data")
ax.set_ylabel("Acidentes por dia")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Exportação do modelo e das previsões

In [ ]:
PASTA_ARTEFATOS = ARQUIVO_ACIDENTES.parents[2] / "ModeloEstatistico" / "artifacts"
PASTA_ARTEFATOS.mkdir(parents=True, exist_ok=True)

caminho_modelo = PASTA_ARTEFATOS / "arimax_volume_acidentes.joblib"
caminho_previsao = PASTA_ARTEFATOS / "previsao_acidentes_diarios.csv"
caminho_avaliacao = PASTA_ARTEFATOS / "avaliacao_encadeada_arimax.csv"

pacote_modelo = {
    "modelo": modelo_final,
    "ordem": ORDEM_ARIMAX,
    "colunas_exogenas": list(exog_final.columns),
    "media_volume_log": media_volume_final,
    "desvio_volume_log": desvio_volume_final,
    "data_referencia": DATA_REFERENCIA,
    "ultima_data_treino": treino_final.index.max(),
    "metricas_validacao": metricas_validacao.to_dict(),
    "metricas_encadeadas": (
        None if metricas_encadeadas is None else metricas_encadeadas.to_dict()
    ),
}
joblib.dump(pacote_modelo, caminho_modelo, compress=3)
previsao_acidentes.to_csv(caminho_previsao, encoding="utf-8-sig")
if mascara_observada.any():
    avaliacao_encadeada.to_csv(caminho_avaliacao, encoding="utf-8-sig")

print(f"Modelo salvo em: {caminho_modelo.resolve()}")
print(f"Previsões salvas em: {caminho_previsao.resolve()}")
if mascara_observada.any():
    print(f"Avaliação encadeada salva em: {caminho_avaliacao.resolve()}")